# ARGUS: YOLOv8 Overhead CCTV Fine-Tuning (Kaggle T4 GPU)

This notebook trains a high-precision **YOLOv8** vehicle detector specifically tuned for overhead surveillance and CCTV traffic cameras using the **UA-DETRAC** dataset.

### Before Running:
1. Under **Input** (right sidebar), click **+ Add Input** -> search **Datasets** for `bratjay/ua-detrac-orig` (make sure you select the **Dataset**, not a Notebook).
2. Under **Settings -> Accelerator**, select **`GPU T4 x2`** (or `GPU T4 x1`), and turn **Internet ON**.
3. Use **Save Version -> Save & Run All (Commit)** to run completely in the background — you do NOT need to keep your laptop open!

## 1. Setup Environment & Verify GPU

In [ ]:
!pip install -q ultralytics pyyaml

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError("No GPU detected! Turn on Settings -> Accelerator -> GPU T4 x2.")

## 2. Auto-Locate Dataset & Convert UA-DETRAC XML Annotations to YOLO Format
Automatically locates `MVI_*.xml` under `/kaggle/input` and fails loudly if the `bratjay/ua-detrac-orig` dataset is missing.

In [ ]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path

DETRAC_CLASS_MAP = {
    "car": 0,
    "van": 0,
    "bus": 1,
    "others": 2,
}

IMG_WIDTH = 960.0
IMG_HEIGHT = 540.0

input_root = Path("/kaggle/input")
mounted = [str(p) for p in input_root.iterdir()] if input_root.exists() else []
print("Mounted in /kaggle/input:", mounted)

# Check direct known paths first (avoids slow rglob across 141k image files)
candidate_xml_dirs = [
    input_root / "ua-detrac-orig" / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML",
    input_root / "datasets" / "bratjay" / "ua-detrac-orig" / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML",
    input_root / "ua-detrac-orig" / "DETRAC-Train-Annotations-XML",
    input_root / "datasets" / "bratjay" / "ua-detrac-orig" / "DETRAC-Train-Annotations-XML",
]
xml_dir = next((d for d in candidate_xml_dirs if d.exists() and any(d.glob("MVI_*.xml"))), None)
if xml_dir is not None:
    xml_files = sorted(xml_dir.glob("MVI_*.xml"))
else:
    all_xmls = sorted(input_root.rglob("MVI_*.xml"))
    train_xmls = [p for p in all_xmls if "Test" not in str(p)]
    xml_files = train_xmls if train_xmls else all_xmls

if not xml_files:
    raise FileNotFoundError(
        f"No UA-DETRAC XML files found in /kaggle/input! Currently mounted: {mounted}. "
        "Please add the DATASET 'bratjay/ua-detrac-orig' via '+ Add Input -> Datasets'."
    )

labels_out_dir = Path("/kaggle/working/labels")
labels_out_dir.mkdir(parents=True, exist_ok=True)
print(f"Found {len(xml_files)} sequence XML files under {xml_files[0].parent}. Converting...")

total_converted = 0
for xml_file in xml_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    seq_name = root.attrib.get("name", xml_file.stem)
    seq_lbl_dir = labels_out_dir / seq_name
    seq_lbl_dir.mkdir(parents=True, exist_ok=True)

    for frame_elem in root.findall("frame"):
        frame_num = int(frame_elem.attrib["num"])
        target_list = frame_elem.find("target_list")
        if target_list is None:
            continue

        lines = []
        for target in target_list.findall("target"):
            box = target.find("box")
            attr = target.find("attribute")
            if box is None:
                continue

            left = float(box.attrib["left"])
            top = float(box.attrib["top"])
            width = float(box.attrib["width"])
            height = float(box.attrib["height"])

            v_type = attr.attrib.get("vehicle_type", "car").lower() if attr is not None else "car"
            cls_id = DETRAC_CLASS_MAP.get(v_type, 0)

            x_center = min(1.0, max(0.0, (left + width / 2.0) / IMG_WIDTH))
            y_center = min(1.0, max(0.0, (top + height / 2.0) / IMG_HEIGHT))
            w_norm = min(1.0, max(0.0, width / IMG_WIDTH))
            h_norm = min(1.0, max(0.0, height / IMG_HEIGHT))

            lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        out_lbl = seq_lbl_dir / f"img{frame_num:05d}.txt"
        out_lbl.write_text("\n".join(lines), encoding="utf-8")
        total_converted += 1

print(f"Finished! Converted {total_converted:,} frames across {len(xml_files)} sequences.")

## 3. Zero-Copy Train / Validation Split (80/20)
Uses Linux symbolic links to connect `/kaggle/working/images` to `/kaggle/input/` without consuming disk quota.

In [ ]:
import random

# Check direct known image roots first (avoids walking 141k JPGs with rglob)
candidate_img_roots = [
    input_root / "ua-detrac-orig" / "DETRAC-Images" / "DETRAC-Images",
    input_root / "datasets" / "bratjay" / "ua-detrac-orig" / "DETRAC-Images" / "DETRAC-Images",
    input_root / "ua-detrac-orig" / "DETRAC-Images",
    input_root / "datasets" / "bratjay" / "ua-detrac-orig" / "DETRAC-Images",
]
img_root = next((d for d in candidate_img_roots if d.exists() and any(d.glob("MVI_*"))), None)
if img_root is not None:
    seq_img_dirs = {p.name: p for p in img_root.iterdir() if p.is_dir() and p.name.startswith("MVI_")}
else:
    seq_img_dirs = {p.name: p for p in input_root.rglob("MVI_*") if p.is_dir()}
if not seq_img_dirs:
    raise FileNotFoundError("No MVI_* image sequence directories found under /kaggle/input!")

train_img_dir = Path("/kaggle/working/images/train")
val_img_dir = Path("/kaggle/working/images/val")
train_lbl_dir = Path("/kaggle/working/labels/train")
val_lbl_dir = Path("/kaggle/working/labels/val")

for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    d.mkdir(parents=True, exist_ok=True)

lbl_seq_names = {d.name for d in labels_out_dir.iterdir() if d.is_dir() and d.name.startswith("MVI_")}
seq_names = sorted(list(lbl_seq_names & set(seq_img_dirs.keys())))
random.seed(42)
random.shuffle(seq_names)

split_idx = int(len(seq_names) * 0.8)
train_seqs = set(seq_names[:split_idx])
val_seqs = set(seq_names[split_idx:])

print(f"Matched {len(seq_names)} sequences -> Train: {len(train_seqs)} | Val: {len(val_seqs)}")

for seq_name in seq_names:
    is_train = seq_name in train_seqs
    dest_img_dir = train_img_dir if is_train else val_img_dir
    dest_lbl_dir = train_lbl_dir if is_train else val_lbl_dir

    src_img_seq = seq_img_dirs[seq_name]
    src_lbl_seq = labels_out_dir / seq_name

    for img_path in src_img_seq.glob("*.jpg"):
        dest_img = dest_img_dir / f"{seq_name}_{img_path.name}"
        if not dest_img.exists():
            os.symlink(str(img_path), str(dest_img))

        lbl_path = src_lbl_seq / f"{img_path.stem}.txt"
        if lbl_path.exists():
            dest_lbl = dest_lbl_dir / f"{seq_name}_{img_path.stem}.txt"
            if not dest_lbl.exists():
                os.symlink(str(lbl_path), str(dest_lbl))

n_train = len(list(train_img_dir.glob("*")))
n_val = len(list(val_img_dir.glob("*")))
print(f"Train images ready: {n_train:,}")
print(f"Val images ready:   {n_val:,}")
assert n_train > 0 and n_val > 0, "Train/val split is empty!"

## 4. Build `data.yaml`

In [ ]:
import yaml

data_yaml = {
    "path": "/kaggle/working",
    "train": "images/train",
    "val": "images/val",
    "nc": 3,
    "names": {
        0: "car",
        1: "bus",
        2: "others",
    },
}

with open("/kaggle/working/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

!cat /kaggle/working/data.yaml

## 5. Fine-Tune YOLOv8n on Tesla T4 GPU (10 Epochs, Resumes from Checkpoint if Present)
Auto-detects `last.pt` (e.g. from Epoch 4) and runs remaining epochs to reach 10 total, or runs 10 epochs fresh.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

ckpt = Path("/kaggle/working/runs/argus-detrac-v1/weights/last.pt")
if ckpt.exists():
    print(f"Found checkpoint {ckpt} — continuing for 6 more epochs to reach 10 total!")
    model = YOLO(str(ckpt))
    target_epochs = 6
else:
    print("Starting fresh 10-epoch fine-tuning from yolov8n.pt...")
    model = YOLO("yolov8n.pt")
    target_epochs = 10

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=target_epochs,
    batch=32,            # Fast training on Tesla T4
    imgsz=640,
    device=0,
    project="/kaggle/working/runs",
    name="argus-detrac-v1",
    exist_ok=True,
    half=True,           # FP16 acceleration
    workers=4,
    patience=10,
    mosaic=1.0,
    degrees=5.0,
    flipud=0.0,
    fliplr=0.5,
    verbose=True,
)

print("Training finished successfully!")

## 6. Package & Export Weights for ARGUS
Packages `best.pt` and training metrics into a single zip file and renders a direct download link.

In [ ]:
!zip -j /kaggle/working/argus_yolov8_best_weights.zip \
    /kaggle/working/runs/argus-detrac-v1/weights/best.pt \
    /kaggle/working/runs/argus-detrac-v1/results.png \
    /kaggle/working/runs/argus-detrac-v1/confusion_matrix.png

from IPython.display import FileLink
print("Done! Click the link below (or download from Output sidebar):")
FileLink("argus_yolov8_best_weights.zip")